# GCN Native Adapter v1

目标：把 `gcn_mse` 从 feature propagation Ridge proxy 推进到 `src/model` 的 native graph adapter。Notebook 只保留实验流程和中文结论；可复用训练、回测、analysis 逻辑都调用 `src/`。外部论文只保存 URL 与自有摘要，不复制全文或第三方代码。

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'src').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model.capabilities import model_capability_table, model_fidelity_summary
from src.model.graph import GCNGraphRegressor
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.training import fit_estimator_for_model, model_adapter_name

OUTPUT_DIR = REPO_ROOT / 'research' / 'gcn_native_adapter_v1'
CARD_DIR = OUTPUT_DIR / 'matrix_card'

outputs = run_quantbench_matrix_smoke(
    card_dir=CARD_DIR,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'market_graph_mse', 'gcn_mse', 'gat_mse'),
)

frame = load_soxl_support_frame(start='2026-01-01', end='2026-06-09')
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame['date'] <= '2026-04-30'].copy()
feature_columns = tuple(dataset.feature_columns[:8])
estimator, constant_prediction = fit_estimator_for_model('gcn_mse', train=train, feature_columns=feature_columns)
capability = model_capability_table(('gcn_mse',)).iloc[0].to_dict()
fidelity = model_fidelity_summary()

assert constant_prediction is None
assert isinstance(estimator, GCNGraphRegressor)
assert estimator.feature_count_ == 8
assert estimator.node_count_ == 8
assert estimator.normalized_adjacency_.shape == (8, 8)
assert hasattr(estimator, 'model_')
assert hasattr(estimator.model_, 'normalized_adjacency')
assert capability['paper_fidelity'] == 'native_graph'
assert capability['is_proxy'] is False
assert model_adapter_name('gcn_mse') == 'gcn_graph_convolution_mse_v1'
assert 'gcn_mse' in fidelity['native_models']
assert 'gcn_mse' not in fidelity['proxy_models']

summary = {
    'schema_version': 'research.gcn_native_adapter_v1',
    'status': 'completed',
    'model': 'gcn_mse',
    'adapter': model_adapter_name('gcn_mse'),
    'estimator': estimator.__class__.__name__,
    'node_count': estimator.node_count_,
    'feature_count': estimator.feature_count_,
    'hidden_size': estimator.hidden_size,
    'paper_fidelity': capability['paper_fidelity'],
    'is_proxy': bool(capability['is_proxy']),
    'full_matrix_native_model_count': len(fidelity['native_models']),
    'full_matrix_proxy_model_count': len(fidelity['proxy_models']),
    'matrix_card': str(CARD_DIR.relative_to(REPO_ROOT)),
    'paper_reference': {
        'source_id': 'arxiv:1609.02907',
        'url': 'https://arxiv.org/abs/1609.02907',
        'usage': '自有摘要和结构假设；实现只复用 graph convolution 传播思想，不复制论文全文或第三方代码。',
    },
    'artifact_links': {
        'analysis_summary': str((CARD_DIR / 'analysis_summary.json').relative_to(REPO_ROOT)),
        'model_comparison': str((CARD_DIR / 'model_comparison.csv').relative_to(REPO_ROOT)),
        'model_run_ledger': str((CARD_DIR / 'model_run_ledger.csv').relative_to(REPO_ROOT)),
        'manifest': str((CARD_DIR / 'analysis_manifest.json').relative_to(REPO_ROOT)),
    },
}
(OUTPUT_DIR / 'notebook_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'research.gcn_native_adapter_v1',
 'status': 'completed',
 'model': 'gcn_mse',
 'adapter': 'gcn_graph_convolution_mse_v1',
 'estimator': 'GCNGraphRegressor',
 'node_count': 8,
 'feature_count': 8,
 'hidden_size': 8,
 'paper_fidelity': 'native_graph',
 'is_proxy': False,
 'full_matrix_native_model_count': 21,
 'full_matrix_proxy_model_count': 11,
 'matrix_card': 'research/gcn_native_adapter_v1/matrix_card',
 'paper_reference': {'source_id': 'arxiv:1609.02907',
  'url': 'https://arxiv.org/abs/1609.02907',
  'usage': '自有摘要和结构假设；实现只复用 graph convolution 传播思想，不复制论文全文或第三方代码。'},
 'artifact_links': {'analysis_summary': 'research/gcn_native_adapter_v1/matrix_card/analysis_summary.json',
  'model_comparison': 'research/gcn_native_adapter_v1/matrix_card/model_comparison.csv',
  'model_run_ledger': 'research/gcn_native_adapter_v1/matrix_card/model_run_ledger.csv',
  'manifest': 'research/gcn_native_adapter_v1/matrix_card/analysis_manifest.json'}}

## 外部参考摘要

- Kipf and Welling, `Semi-Supervised Classification with Graph Convolutional Networks`：核心结构是用归一化邻接矩阵传播节点表征，并用浅层神经网络完成节点层预测。参考：https://arxiv.org/abs/1609.02907
- 本仓库实现：SOXL 日频监督任务没有天然多资产节点标签，本轮先把特征列视为 feature nodes，用训练窗口相关性构建邻接矩阵，做 compact graph convolution regression；后续若引入 SOXX/VIX/QQQ/SPY 或行业成分真实关系图，应替换 adjacency provider，而不是改 notebook 里的临时代码。

## 中文结论

- 做了什么：在 `src/model.graph` 增加 `GCNGraphRegressor`，把 `gcn_mse` 从 `GraphFeatureRidgeRegressor(graph_kind='gcn')` proxy 升级为 `native_graph`。实现使用训练窗口特征相关性构建归一化邻接矩阵，两层 graph convolution pooling 后回归预测。
- 核心指标：本 notebook 训练并回测 `ridge_mse`、`market_graph_mse`、`gcn_mse`、`gat_mse`，输出 `model_comparison.csv`、`daily_backtest.csv`、`model_run_ledger.csv`、`qlib_risk_metrics.csv`、`model_effect_deltas.csv`、`model_regime_window_diagnostics.csv` 和 manifest。具体收益、回撤、Sharpe 以 `notebook_summary.json` 与 `matrix_card/analysis_summary.json` 为准。
- regime 差异：本轮复用 `src/analysis` 的 qlib risk、window/regime、model effect delta 和模型级 regime/window 诊断；GCN 的价值要看 stress/downtrend 下传播后的预测是否比 market/gat proxy 更稳，而不是只看全区间收益。
- 是否晋升：否。`native_graph` 只表示实现形态已从 proxy 进入稳定训练接口；production 晋升仍需要真实支持宇宙关系图、长历史、多 regime、成本压力和 paper/sim replay。
- 风险：当前 adjacency 来自短训练窗口特征相关性，可能过拟合；不是完整资产图或动态图 GNN。
- 下一步：刷新全模型 ledger，使 native/proxy 计数同步；继续替换 `market_graph_mse`、`gat_mse`、`rgcn_mse`、`hypergraph_mse`、`thgnn_mse` 等 graph proxy。